# 🚀 PySpark - 100+ Exemplos Práticos## Guia Completo para Agronegócio - Copy & Paste Ready> **Este manual contém mais de 100 exemplos prontos para copiar e usar.**> Todos os exemplos são contextualizados para o agronegócio (plantio, colheita, produtividade, etc.)---## 📑 ÍNDICE COMPLETO### 🔰 Parte 1: Fundamentos (23 exemplos)- [1.1 Setup e Configuração](#11-setup-e-configuração) - 5 exemplos- [1.2 Criando DataFrames](#12-criando-dataframes) - 10 exemplos- [1.3 Explorando Dados](#13-explorando-dados) - 8 exemplos### 📌 Parte 2: Seleção e Filtros (33 exemplos)- [2.1 Selecionar Colunas](#21-selecionar-colunas) - 12 exemplos- [2.2 Filtrar Dados](#22-filtrar-dados) - 15 exemplos- [2.3 Remover Duplicatas](#23-remover-duplicatas) - 6 exemplos### ⚡ Parte 3: Transformações (67 exemplos)- [3.1 Adicionar/Modificar Colunas](#31-adicionarmodificar-colunas) - 10 exemplos- [3.2 Trabalhar com Textos](#32-trabalhar-com-textos) - 15 exemplos- [3.3 Trabalhar com Números](#33-trabalhar-com-números) - 12 exemplos- [3.4 Trabalhar com Datas](#34-trabalhar-com-datas) - 20 exemplos- [3.5 Condicionais (IF/WHEN)](#35-condicionais) - 10 exemplos### 📊 Parte 4: Agregações (35 exemplos)- [4.1 Agregações Simples](#41-agregações-simples) - 8 exemplos- [4.2 GroupBy Avançado](#42-groupby-avançado) - 12 exemplos- [4.3 Window Functions](#43-window-functions) - 15 exemplos### 🔗 Parte 5: Joins e Unions (15 exemplos)- [5.1 Joins](#51-joins) - 10 exemplos- [5.2 Unions](#52-unions) - 5 exemplos### 🌾 Parte 6: Casos de Uso do Agronegócio (28 exemplos)- [6.1 Análise de Plantio](#61-análise-de-plantio) - 10 exemplos- [6.2 Controle de Qualidade](#62-controle-de-qualidade) - 8 exemplos- [6.3 Produtividade](#63-produtividade) - 10 exemplos**TOTAL: 200+ EXEMPLOS PRÁTICOS!**---# 🔰 PARTE 1: FUNDAMENTOS## 1.1 Setup e Configuração### Exemplo 1: Imports Completos

In [ ]:
# ============================================================================# COPIE ESTE BLOCO NO INÍCIO DE TODO NOTEBOOK PYSPARK# ============================================================================from pyspark.sql import SparkSession, Windowfrom pyspark.sql import functions as Ffrom pyspark.sql.types import *from datetime import datetime, timedelta# Funções mais usadas (importar diretamente economiza digitação)from pyspark.sql.functions import (    col, lit, when, count, sum, avg, max, min,    year, month, dayofmonth, date_format, to_date,    upper, lower, trim, concat, regexp_replace,    round, abs, sqrt, lag, lead, row_number, rank,    array, explode, size)print("✅ Imports carregados!")

---### Exemplo 2: Criar Spark Session

In [ ]:
# ============================================================================# CRIAR SPARK SESSION (SÓ SE NÃO EXISTIR)# ============================================================================# No Databricks, 'spark' já existe automaticamente# Mas se precisar criar manualmente:try:    # Testa se spark existe    spark    print("✅ Spark Session já existe!")except:    # Cria nova sessão    spark = SparkSession.builder \        .appName("ProcessamentoAgricola") \        .config("spark.sql.adaptive.enabled", "true") \        .config("spark.sql.shuffle.partitions", "200") \        .getOrCreate()    print(f"✅ Spark {spark.version} iniciado!")# Verificaspark.catalog.listDatabases()

**💡 Explicação**:- `appName`: Nome da aplicação (aparece nos logs)- `adaptive.enabled`: Otimização automática- `shuffle.partitions`: Número de partições após shuffle (ajuste baseado no tamanho dos dados)---### Exemplo 3: Conectar Azure Data Lake

In [ ]:
# ============================================================================# CONFIGURAR ACESSO AO AZURE DATA LAKE# ============================================================================# Suas credenciaisstorage_account = "seustorage"container = "bronze"app_id = "sua-aplicacao-id"app_secret = "sua-chave-secreta"tenant_id = "seu-tenant-id"# Configuraçãospark.conf.set(    f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net",    "OAuth")spark.conf.set(    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net",    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")spark.conf.set(    f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net",    app_id)spark.conf.set(    f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net",    app_secret)spark.conf.set(    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net",    f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")# Caminho base para usarbase_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net"# Testedbutils.fs.ls(base_path)print(f"✅ Conectado ao Azure Data Lake: {base_path}")

---### Exemplo 4: Conectar Synapse Analytics

In [ ]:
# ============================================================================# LER DO AZURE SYNAPSE ANALYTICS# ============================================================================# Configuração do Synapsesynapse_config = {    "server": "seusynapse.sql.azuresynapse.net",    "database": "DW_Agricola",    "user": "usuario",    "password": "senha"}jdbc_url = f"jdbc:sqlserver://{synapse_config['server']}:1433;database={synapse_config['database']}"# Função helper para ler tabelasdef ler_synapse(tabela):    """Lê tabela do Synapse e retorna DataFrame Spark."""    return spark.read \        .format("jdbc") \        .option("url", jdbc_url) \        .option("dbtable", tabela) \        .option("user", synapse_config['user']) \        .option("password", synapse_config['password']) \        .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver") \        .load()# Usodf_plantio = ler_synapse("fato_plantio")df_talhoes = ler_synapse("dim_talhoes")print(f"✅ Plantio: {df_plantio.count():,} registros")print(f"✅ Talhões: {df_talhoes.count():,} registros")

---### Exemplo 5: Ler SharePoint (via Pandas)

In [ ]:
# ============================================================================# LER ARQUIVO DO SHAREPOINT# ============================================================================# Para arquivos pequenos (< 100MB), use pandas como intermediárioimport pandas as pdfrom office365.sharepoint.client_context import ClientContextfrom office365.runtime.auth.authentication_context import AuthenticationContextdef ler_sharepoint_excel(site_url, file_path, username, password, sheet_name=0):    """    Lê Excel do SharePoint e retorna Spark DataFrame.    Args:        site_url: URL do site SharePoint        file_path: Caminho do arquivo (ex: "/sites/Dados/arquivo.xlsx")        username: Email do usuário        password: Senha        sheet_name: Nome da aba ou índice (padrão: 0)    Returns:        DataFrame Spark    """    # Autenticação    auth_ctx = AuthenticationContext(site_url)    auth_ctx.acquire_token_for_user(username, password)    ctx = ClientContext(site_url, auth_ctx)    # Download    response = ctx.web.get_file_by_server_relative_url(file_path).download().execute_query()    # Pandas    pdf = pd.read_excel(response.content, sheet_name=sheet_name)    # Spark    df = spark.createDataFrame(pdf)    print(f"✅ Lido {len(pdf):,} linhas de {file_path}")    return df# Usodf = ler_sharepoint_excel(    site_url="https://empresa.sharepoint.com/sites/Agricola",    file_path="/sites/Agricola/Documentos/plantio_2024.xlsx",    username="seu@email.com",    password="senha",    sheet_name="Dados")

---## 1.2 Criando DataFrames### Exemplo 6: De Lista de Tuplas

In [ ]:
# ============================================================================# CRIAR DATAFRAME DE LISTA DE TUPLAS# ============================================================================# Dados de exemplo: plantiodados = [    ("T001", "RB867515", 45.5, "2024-03-15", 88.5, 148.2),    ("T002", "RB966928", 38.2, "2024-03-16", 92.1, 151.5),    ("T003", "CTC4", 52.0, "2024-03-17", 85.3, 145.8),    ("T004", "SP81-3250", 28.7, "2024-03-18", 78.4, 140.5),]# Colunascolunas = ["talhao", "variedade", "area_ha", "data_plantio", "tch", "atr"]# Cria DataFramedf = spark.createDataFrame(dados, colunas)# Ajusta tipo de datadf = df.withColumn("data_plantio", to_date("data_plantio", "yyyy-MM-dd"))df.show()df.printSchema()

**💡 Quando usar**: Dados pequenos, testes rápidos, exemplos.---### Exemplo 7: De Dicionários (Mais Legível)

In [ ]:
# ============================================================================# CRIAR DATAFRAME DE LISTA DE DICIONÁRIOS# ============================================================================# Mais legível que tuplas!dados = [    {        "talhao": "T001",        "variedade": "RB867515",        "area_ha": 45.5,        "data": "2024-03-15",        "tch": 88.5,        "atr": 148.2    },    {        "talhao": "T002",        "variedade": "RB966928",        "area_ha": 38.2,        "data": "2024-03-16",        "tch": 92.1,        "atr": 151.5    },]df = spark.createDataFrame(dados)# Converte datadf = df.withColumn("data", to_date("data", "yyyy-MM-dd"))df.show()

**💡 Vantagem**: Não precisa lembrar ordem das colunas!---### Exemplo 8: Com Schema Explícito (Recomendado)

In [ ]:
# ============================================================================# CRIAR DATAFRAME COM SCHEMA EXPLÍCITO (PRODUÇÃO)# ============================================================================# Define schema (tipos e nullable)schema = StructType([    StructField("talhao", StringType(), nullable=False),         # NOT NULL    StructField("variedade", StringType(), nullable=False),    StructField("area_ha", DoubleType(), nullable=True),        # NULLABLE    StructField("data_plantio", DateType(), nullable=False),    StructField("tch", DoubleType(), nullable=True),    StructField("atr", DoubleType(), nullable=True),    StructField("observacao", StringType(), nullable=True)])# Dadosdados = [    ("T001", "RB867515", 45.5, datetime(2024, 3, 15), 88.5, 148.2, None),    ("T002", "RB966928", 38.2, datetime(2024, 3, 16), 92.1, 151.5, "OK"),]df = spark.createDataFrame(dados, schema)df.printSchema()# root#  |-- talhao: string (nullable = false)#  |-- variedade: string (nullable = false)#  |-- area_ha: double (nullable = true)#  |-- data_plantio: date (nullable = false)#  |-- tch: double (nullable = true)#  |-- atr: double (nullable = true)#  |-- observacao: string (nullable = true)

**💡 Quando usar**: Produção, validação de dados, performance.---### Exemplo 9: Ler CSV (Todas as Opções)

In [ ]:
# ============================================================================# LER CSV COM TODAS AS OPÇÕES POSSÍVEIS# ============================================================================df = spark.read \    .format("csv") \    .option("header", "true") \              # Primeira linha = cabeçalho    .option("inferSchema", "true") \         # Detecta tipos (lento, mas útil)    .option("sep", ";") \                    # Separador (padrão: ,)    .option("encoding", "UTF-8") \           # Encoding    .option("quote", '"') \                  # Caractere de aspas    .option("escape", "\\") \                # Escape    .option("nullValue", "NULL") \           # Como NULL está no arquivo    .option("dateFormat", "dd/MM/yyyy") \    # Formato de data    .option("timestampFormat", "dd/MM/yyyy HH:mm:ss") \    .option("mode", "PERMISSIVE") \          # O que fazer com linhas ruins    .option("columnNameOfCorruptRecord", "_corrupt_record") \    .option("multiLine", "true") \           # Se campos têm quebras de linha    .option("ignoreLeadingWhiteSpace", "true") \    .option("ignoreTrailingWhiteSpace", "true") \    .load("dados_plantio.csv")# Modos disponíveis:# - PERMISSIVE (padrão): Mantém linhas ruins, coloca NULL nos campos inválidos# - DROPMALFORMED: Remove linhas ruins# - FAILFAST: Para execução se encontrar linha ruimprint(f"✅ {df.count():,} linhas carregadas")

**💡 Dica**: Use `inferSchema=false` e defina schema manualmente para produção (mais rápido).---### Exemplo 10: Ler Parquet (Recomendado)

In [ ]:
# ============================================================================# LER PARQUET (FORMATO RECOMENDADO)# ============================================================================# Leitura simplesdf = spark.read.parquet("caminho/dados.parquet")# Com múltiplos arquivosdf = spark.read.parquet(    "caminho/arquivo1.parquet",    "caminho/arquivo2.parquet",    "caminho/arquivo3.parquet")# Pasta inteiradf = spark.read.parquet("caminho/pasta_com_parquets/")# Com schema evolution (permite schemas diferentes)df = spark.read \    .option("mergeSchema", "true") \    .parquet("caminho/")print(f"✅ {df.count():,} registros carregados")df.printSchema()

**💡 Vantagens do Parquet**:- 10-100x menor que CSV- Leitura muito mais rápida- Preserva tipos de dados- Suporta compressão---Continuo com mais exemplos nos próximos blocos! Quer que eu continue agora ou você prefere revisar e depois peço para continuar?